# AN-RA Master Operator Notebook

Run this notebook as the Colab control surface for An-Ra.

It is no longer only a training notebook. It is the full operator loop:

| Cell | Purpose |
| --- | --- |
| 1 | Session configuration and component flags |
| 2 | GPU, RAM, Drive, and dependency setup |
| 3 | Clone or update the repository |
| 4 | Select and merge owner training data |
| 5 | Restore artifacts and run preflight checks |
| 6 | Apply feature flags and print the system report |
| 7 | Run training or evaluation |
| 8 | Inspect telemetry and performance scorecard |
| 9 | Sync reports/checkpoints back to Drive |
| 10 | Launch API/UI surfaces |

Operating rule: every serious run should end with a report, telemetry summary, and saved artifacts.


In [ ]:
# CELL 1 - CONFIG
# Edit this cell first. Everything else should be runnable top-to-bottom.

SESSION_MINUTES = 90

# MODEL SIZE
# "25m" -> current architecture (512 embd, 8 layers) - dev / test
# "1b"  -> V2_1B_FRONTIER (1536 embd, 36 layers, 16 heads) - frontier run
MODEL_SIZE = "1b"
OUROBOROS_MINUTES = 10
TRAINING_MODE = "session"  # "status", "session", "train", or "eval"

# TRAINING_PRESET - sets MODEL_SIZE + SESSION_MINUTES + TRAINING_MODE at once
# Set to None to use individual settings above
TRAINING_PRESET = "session_1b"   # None, "dev", "session_1b", "deep_1b"

_PRESETS = {
    "dev":        {"MODEL_SIZE": "25m", "SESSION_MINUTES": 30,
                   "TRAINING_MODE": "status"},
    "session_1b": {"MODEL_SIZE": "1b",  "SESSION_MINUTES": 90,
                   "TRAINING_MODE": "session"},
    "deep_1b":    {"MODEL_SIZE": "1b",  "SESSION_MINUTES": 180,
                   "TRAINING_MODE": "train"},
}
if TRAINING_PRESET and TRAINING_PRESET in _PRESETS:
    for _k, _v in _PRESETS[TRAINING_PRESET].items():
        globals()[_k] = _v
    print(f"Preset '{TRAINING_PRESET}' applied")

MODEL_SIZE_LABEL = {
    "25m": "25M params - fast, dev/test",
    "1b": "1B params - frontier, RTX6000Ada/A100 required",
}.get(MODEL_SIZE, "unknown")
print(f"Model size: {MODEL_SIZE_LABEL}")
RUN_TESTS = False           # set True for a slower full verification pass
RUN_REPORT_BEFORE_TRAIN = True
RUN_REPORT_AFTER_TRAIN = True

REPO_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
REPO_PATH = "/content/An-Ra-the-new-AGI"
DRIVE_ANRA_DIR = "/content/drive/MyDrive/AnRa"
DRIVE_V2_DIR = f"{DRIVE_ANRA_DIR}/v2"
DRIVE_V2_CHECKPOINTS = f"{DRIVE_V2_DIR}/checkpoints"
DRIVE_REPORTS = f"{DRIVE_V2_DIR}/reports"
DRIVE_DATASET = f"{DRIVE_ANRA_DIR}/anra_training.txt"
DRIVE_DATASET_LEGACY = f"{DRIVE_ANRA_DIR}/anra_dataset_v6_1.txt"
LOCAL_MERGED_DATA = "/content/anra_merged_data/anra_training.txt"

# Component flags. Leave everything True unless you are doing ablation or debugging.
COMPONENT_FLAGS = {
    "brain": True,
    "tokenizer": True,
    "data_mix": True,
    "training_loop": True,
    "evaluation": True,
    "runtime": True,
    "api_web": True,
    "identity": True,
    "memory": True,
    "phase2_memory": True,
    "goals": True,
    "agent_loop": True,
    "master_system": True,
    "self_improvement": True,
    "self_modification": True,
    "ouroboros": True,
    "ghost_memory": True,
    "symbolic_bridge": True,
    "sovereignty": True,
}

SELECTED_DATASETS = []
TRAINING_EXIT_CODE = None

print("AN-RA SESSION PLAN")
print("=" * 60)
print(f"Mode:              {TRAINING_MODE}")
print(f"Session minutes:   {SESSION_MINUTES}")
print(f"Model size:        {MODEL_SIZE_LABEL}")
print(f"Ouroboros minutes: {OUROBOROS_MINUTES}")
print(f"Run tests:         {RUN_TESTS}")
print(f"Repo:              {REPO_PATH}")
print(f"Drive root:        {DRIVE_ANRA_DIR}")
print("Enabled components:", sum(1 for v in COMPONENT_FLAGS.values() if v), "/", len(COMPONENT_FLAGS))


In [ ]:
# CELL 2 - ENVIRONMENT SETUP
import os, sys, subprocess
from google.colab import drive

print("HARDWARE")
print("=" * 60)
try:
    import psutil
    ram = psutil.virtual_memory()
    print(f"RAM: {ram.available / 1024**3:.1f} GB free / {ram.total / 1024**3:.1f} GB total")
except Exception as exc:
    print("RAM: unavailable", exc)

try:
    import torch
    print(f"Torch:  {torch.__version__}")
    print(f"CUDA:   {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        d = torch.cuda.get_device_properties(0)
        vram_gb = d.total_memory / 1e9
        print(f"GPU:    {d.name}")
        print(f"VRAM:   {vram_gb:.0f} GB")
        if MODEL_SIZE == "1b" and vram_gb < 20:
            print(f"WARNING: {vram_gb:.0f}GB VRAM is tight for 1B. Gradient checkpointing will activate.")
        elif MODEL_SIZE == "1b":
            print(f"OK: {vram_gb:.0f}GB VRAM is sufficient for 1B training.")
except Exception as exc:
    print("Torch check failed:", exc)

# Install Muon optimizer if not present
print("\nChecking Muon optimizer...")
try:
    import muon
    print("Muon: already installed")
except ImportError:
    print("Installing Muon...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/KellerJordan/Muon"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("Muon: installed")
    else:
        print("Muon install failed (AdamW fallback will be used):", result.stderr[-200:])

# Mount Drive
print("\nMounting Drive...")
drive.mount("/content/drive", force_remount=False)
print("Drive mounted")


In [ ]:
# CELL 3 - REPOSITORY SETUP
import os
import sys
import shutil
import subprocess

print("REPOSITORY")
print("=" * 60)
if os.path.isdir(REPO_PATH):
    print("Repo exists; updating current checkout")
    result = subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_PATH, capture_output=True, text=True)
    if result.returncode != 0:
        print("Fast-forward pull failed; recloning for a clean Colab workspace")
        print(result.stderr[-1000:])
        shutil.rmtree(REPO_PATH)
        result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_PATH], capture_output=True, text=True)
else:
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_PATH], capture_output=True, text=True)

if result.returncode != 0:
    raise RuntimeError(f"Repository setup failed:\n{result.stderr[-2000:]}")

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
with open("/tmp/anra_repo_path.txt", "w", encoding="utf-8") as f:
    f.write(REPO_PATH)

print("Repo ready:", REPO_PATH)
print("Current commit:")
subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_PATH)


In [ ]:
# CELL 4 - DATA SELECTOR AND MERGE
import os
import glob
import json

# Optional: download HuggingFace datasets if training_data is sparse
base_exists = os.path.exists(f"{REPO_PATH}/training_data/base_corpus.txt")
dfc_exists = os.path.exists(f"{REPO_PATH}/training_data/frontier_dfc.jsonl")
base_size_mb = os.path.getsize(f"{REPO_PATH}/training_data/base_corpus.txt")/1024**2 \
               if base_exists else 0

if not base_exists or base_size_mb < 100:
    print("Base corpus missing or small. Run to download:")
    print(f"  !cd {REPO_PATH} && pip install datasets -q && \\")
    print(f"  python scripts/download_training_data.py --bucket base")
    print()
else:
    print(f"Base corpus: {base_size_mb:.0f} MB ✅")

if not dfc_exists:
    print("DFC data missing. Run:")
    print(f"  !cd {REPO_PATH} && python scripts/download_training_data.py --bucket science")
else:
    dfc_lines = sum(1 for _ in open(f"{REPO_PATH}/training_data/frontier_dfc.jsonl"))
    print(f"DFC examples: {dfc_lines:,} ✅")

print("DATA DISCOVERY")
print("=" * 60)
all_txt = glob.glob(f"{DRIVE_ANRA_DIR}/**/*.txt", recursive=True)
all_jsonl = glob.glob(f"{DRIVE_ANRA_DIR}/**/*.jsonl", recursive=True)
all_files = sorted(set(all_txt + all_jsonl))

if not all_files:
    print("No .txt or .jsonl files found under Drive/AnRa")
    SELECTED_DATASETS = []
else:
    for i, fpath in enumerate(all_files):
        size_mb = os.path.getsize(fpath) / 1024**2
        print(f"[{i:02d}] {os.path.relpath(fpath, DRIVE_ANRA_DIR):<70} {size_mb:8.2f} MB")
    print("\nSelect files by index, comma-separated, or type 'all'. Press Enter for default anra_training.txt.")
    raw = input("Selection: ").strip()
    if raw.lower() == "all":
        indices = list(range(len(all_files)))
    elif raw:
        indices = [int(x.strip()) for x in raw.split(",") if x.strip().isdigit()]
    else:
        indices = []
    SELECTED_DATASETS = [all_files[i] for i in indices if 0 <= i < len(all_files)]
    if not SELECTED_DATASETS and os.path.exists(DRIVE_DATASET):
        SELECTED_DATASETS = [DRIVE_DATASET]

print("\nMERGE")
print("=" * 60)
os.makedirs(os.path.dirname(LOCAL_MERGED_DATA), exist_ok=True)
if SELECTED_DATASETS:
    with open(LOCAL_MERGED_DATA, "w", encoding="utf-8") as out:
        for fpath in SELECTED_DATASETS:
            try:
                if fpath.endswith(".jsonl"):
                    for line in open(fpath, encoding="utf-8", errors="replace"):
                        try:
                            obj = json.loads(line)
                        except Exception:
                            continue
                        for key in ["text", "content", "prompt", "response", "output", "answer"]:
                            value = obj.get(key)
                            if isinstance(value, str) and value.strip():
                                out.write(value.strip() + "\n\n")
                                break
                else:
                    out.write(open(fpath, encoding="utf-8", errors="replace").read().strip() + "\n\n")
                print("merged", os.path.basename(fpath))
            except Exception as exc:
                print("failed", fpath, exc)
    DRIVE_DATASET = LOCAL_MERGED_DATA
    print("Merged dataset:", LOCAL_MERGED_DATA, f"({os.path.getsize(LOCAL_MERGED_DATA) / 1024**2:.2f} MB)")
else:
    print("No selected dataset. Training will use repo bootstrap data if available.")


In [ ]:
# CELL 5 - ARTIFACT RESTORE AND PREFLIGHT
import os
import shutil
import subprocess
import sys
from pathlib import Path

print("ARTIFACT RESTORE")
print("=" * 60)

repo = Path(REPO_PATH)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

local_data = repo / "training_data" / "anra_training.txt"
local_data.parent.mkdir(parents=True, exist_ok=True)

source = None
for candidate in [DRIVE_DATASET, DRIVE_DATASET_LEGACY]:
    if candidate and os.path.exists(candidate):
        source = candidate
        break
if source:
    shutil.copy2(source, local_data)
    print("Dataset copied:", source, "->", local_data)
else:
    print("No Drive dataset copied; repo data will be used if present")

from anra_paths import inject_all_paths, get_v2_checkpoint
inject_all_paths()
from training.v2_runtime import restore_v2_artifact, _read_step

restored = {
    "brain": restore_v2_artifact("brain"),
    "tokenizer": restore_v2_artifact("tokenizer"),
    "identity": restore_v2_artifact("identity"),
    "ouroboros": restore_v2_artifact("ouroboros"),
}

if restored["brain"]:
    step = _read_step(get_v2_checkpoint("brain"))
    print(f"Brain checkpoint ready: step {step}")
else:
    print("No brain checkpoint found; training will start from step 0")

print("Tokenizer:", "ready" if restored["tokenizer"] else "using repo default/build path")

# Restore HAL state (so emotional continuity persists across sessions)
hal_drive = f"{DRIVE_ANRA_DIR}/logs/hal_state.json"
hal_local = repo / "state" / "hal_state.json"
hal_local.parent.mkdir(parents=True, exist_ok=True)
if os.path.exists(hal_drive) and not hal_local.exists():
    shutil.copy2(hal_drive, hal_local)
    print("HAL state restored from Drive")
elif not os.path.exists(hal_drive):
    print("No HAL state on Drive - fresh emotional init")

print("\nPREFLIGHT")
print("=" * 60)
checks = [
    ((repo / "anra.py").exists(), "anra.py present"),
    ((repo / "anra_paths.py").exists(), "anra_paths.py present"),
    ((repo / "runtime" / "system_registry.py").exists(), "system registry present"),
    ((repo / "engine" / "report.py").exists(), "engineering report module present"),
    ((repo / "tokenizer" / "tokenizer_v3.json").exists(), "tokenizer_v3 present"),
    (local_data.exists(), "training data present"),
]
failed = False
for ok, label in checks:
    print(f"{'PASS' if ok else 'FAIL'} - {label}")
    failed = failed or not ok
if failed:
    raise RuntimeError("Preflight failed. Fix missing artifacts before continuing.")

print("\nRegistry smoke:")
subprocess.run([sys.executable, "-c", "from runtime.system_registry import component_registry; print(len(component_registry()), 'components')"], cwd=REPO_PATH)

if MODEL_SIZE == "1b":
    print()
    print("1B MODEL VRAM CHECK")
    try:
        import torch
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            vram_gb = props.total_memory / 1024**3
            gpu_name = props.name
            print(f"  GPU   : {gpu_name}")
            print(f"  VRAM  : {vram_gb:.1f} GB")
            if vram_gb >= 40:
                print("  OK: Comfortable 1B training")
            elif vram_gb >= 20:
                print("  WARN: Tight - grad checkpointing forced, batch=2")
            else:
                raise SystemExit(
                    f"\n1B ABORTED: {vram_gb:.1f}GB VRAM insufficient.\n"
                    f"Need minimum 20GB.\n"
                    f"Recommended: RTX 6000 Ada (48GB) on io.net at $0.97/hr."
                )
        else:
            print("  WARN: No GPU - 1B will be very slow")
    except SystemExit:
        raise
    except Exception as _vram_e:
        print(f"  VRAM check failed: {_vram_e}")


In [ ]:
# CELL 6 - FEATURE FLAGS AND SYSTEM REPORT
import os
import sys
import subprocess

print("APPLY FEATURE FLAGS")
print("=" * 60)
flag_code = """
from engine.feature_flags import set_flag, load_flags
flags = %r
for name, enabled in flags.items():
    set_flag(name, bool(enabled))
print(load_flags())
""" % COMPONENT_FLAGS
result = subprocess.run([sys.executable, "-c", flag_code], cwd=REPO_PATH, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(result.stderr[-1000:])

if RUN_REPORT_BEFORE_TRAIN:
    print("\nSYSTEM REPORT")
    print("=" * 60)
    process = subprocess.run([sys.executable, "anra.py", "--report"], cwd=REPO_PATH, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(process.stdout[-6000:])
    if process.returncode != 0:
        raise RuntimeError("anra.py --report failed")


In [ ]:
# CELL 7 - TRAINING OR EVALUATION RUN
import os
import sys
import time
import subprocess

print("RUN")
print("=" * 60)
cmd = [
    sys.executable, "-m", "training.train_unified",
    "--mode", TRAINING_MODE,
    "--model-size", MODEL_SIZE,
]
if TRAINING_MODE not in {"status", "eval"}:
    cmd.extend([
        "--session-minutes", str(SESSION_MINUTES),
        "--ouroboros_minutes", str(OUROBOROS_MINUTES),
    ])
print("Command:", " ".join(cmd))

env = os.environ.copy()
env["PYTHONPATH"] = REPO_PATH
start = time.time()
process = subprocess.Popen(cmd, cwd=REPO_PATH, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()
TRAINING_EXIT_CODE = process.returncode
elapsed = (time.time() - start) / 60
print("\nRun finished:", TRAINING_EXIT_CODE, f"elapsed={elapsed:.1f} min")
if TRAINING_EXIT_CODE != 0:
    raise RuntimeError(f"Training/eval command failed with code {TRAINING_EXIT_CODE}")


In [ ]:
# CELL 7b - RLVR EVAL (code verification benchmark)
# Run after training to measure verifier-shaped reward signal quality.
import subprocess, sys, os

print("RLVR / VERIFIER EVAL")
print("=" * 60)

env = os.environ.copy()
env["PYTHONPATH"] = REPO_PATH

rlvr_code = """
import sys
sys.path.insert(0, '%(repo)s')
from training.rlvr import RLVRTrainer
from training.verifier import run_verifier_batch
from training.v2_runtime import build_v2_model, load_or_build_v2_tokenizer, load_checkpoint, canonical_v2_checkpoint
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = load_or_build_v2_tokenizer()
model = build_v2_model(vocab_size=tokenizer.vocab_size)
ckpt = canonical_v2_checkpoint("brain")
if ckpt.exists():
    load_checkpoint(model, None, None, None, ckpt, device=device, strict=False)
model = model.to(device).eval()

# Quick sanity: 5 code generation probes
PROBES = [
    ("Write a Python function that returns the nth Fibonacci number.", "fibonacci"),
    ("Write a function that checks if a string is a palindrome.", "palindrome"),
    ("Write a function that returns the factorial of n.", "factorial"),
    ("Write a binary search function.", "binary_search"),
    ("Write a function to flatten a nested list.", "flatten"),
]
passed = 0
for prompt, label in PROBES:
    from generate import generate, GenerationConfig
    cfg = GenerationConfig(strategy="nucleus", max_tokens=150, use_think_tokens=True)
    out = generate(prompt, model, tokenizer, cfg, device=device)
    result = run_verifier_batch([out], [prompt])
    ok = result.get("passed", False)
    passed += int(ok)
    print(f"  {label:<20} {'PASS' if ok else 'FAIL'}")

print(f"\nRLVR probe score: {passed}/{len(PROBES)}")
""" % {"repo": REPO_PATH}

r = subprocess.run([sys.executable, "-c", rlvr_code], cwd=REPO_PATH, env=env,
                   capture_output=True, text=True, timeout=120)
print(r.stdout)
if r.stderr:
    print(r.stderr[-500:])


In [ ]:
# CELL 8 - METRICS
import os
import sys
import subprocess

print("COMPONENT SCORECARD")
print("=" * 60)
score_code = f"""
import sys
sys.path.insert(0, '{REPO_PATH}')
from agents.supervisor import SupervisorAgent

sup = SupervisorAgent(model_size='{MODEL_SIZE}')
summary = sup.end_session()
pushed = sup.push_scorecard_to_drive(summary)

print()
print('COMPONENT METRICS')
print('-' * 60)
for name, m in sorted(summary.raw_metrics.items()):
    sr   = m.get('success_rate', 0)
    lat  = m.get('avg_latency_ms', 0)
    calls = m.get('calls', 0)
    flag = '⚠️ ' if any(name in f for f in summary.components_flagged) else '✅ '
    print(f'  {{flag}}{{name:<28}} calls={{calls:>4}}  sr={{sr:.2f}}  lat={{lat:.0f}}ms')

if summary.deltas:
    print()
    print('DELTAS vs PREVIOUS SESSION')
    print('-' * 60)
    for comp, d in sorted(summary.deltas.items()):
        parts = [f'{{k}}={{v:+.3f}}' for k,v in d.items()]
        print(f'  {{comp:<28}} {{"  ".join(parts)}}')

if summary.recommendations:
    print()
    print('RECOMMENDATIONS')
    print('-' * 60)
    for r in summary.recommendations:
        print(f'  → {{r}}')

print()
print(f'Scorecard Drive push: {{"✅ OK" if pushed else "⚠️  local only"}}')
print(f'Run ID: {{summary.run_id}}')
"""
result = subprocess.run([sys.executable, "-c", score_code], cwd=REPO_PATH, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr[-1000:])

if RUN_REPORT_AFTER_TRAIN:
    print("\nPOST-RUN REPORT")
    print("=" * 60)
    process = subprocess.run([sys.executable, "anra.py", "--report"], cwd=REPO_PATH, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(process.stdout[-6000:])
    if process.returncode != 0:
        raise RuntimeError("post-run report failed")

if RUN_TESTS:
    print("\nTESTS")
    print("=" * 60)
    process = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-x", "-q"], cwd=REPO_PATH, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(process.stdout[-8000:])
    if process.returncode != 0:
        raise RuntimeError("tests failed")


In [ ]:
# CELL 9 - POST-SESSION SCORECARD TO DRIVE
import os
import sys
import subprocess

print("POST-SESSION SCORECARD")
print("=" * 60)
post_code = f"""
from agents.supervisor import SupervisorAgent
from engine.metric_bus import get_metric_bus

bus = get_metric_bus()
run_data = bus.finalize()
print(f"Run ID: {{run_data['run_id']}}")
print(f"Duration: {{run_data.get('duration_minutes', 0):.1f}} min")
print(f"Components active: {{list(run_data['components'].keys())}}")
if run_data.get('deltas'):
    print("Deltas vs last run:")
    for comp, d in run_data['deltas'].items():
        print(f"  {{comp}}: {{d}}")

supervisor = SupervisorAgent(model_size={MODEL_SIZE!r})
summary = supervisor.end_session()
supervisor.push_scorecard_to_drive(summary)
print("\nScorecard written. Open Drive -> AnRa/logs/scorecards/ to review.")
"""
result = subprocess.run([sys.executable, "-c", post_code], cwd=REPO_PATH, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr[-2000:])
if result.returncode != 0:
    raise RuntimeError("post-session scorecard failed")


In [ ]:
# CELL 10 - SYNC REPORTS AND CHECKPOINTS TO DRIVE
import os
import shutil
import sys
from pathlib import Path

print("SYNC TO DRIVE")
print("=" * 60)

repo = Path(REPO_PATH)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from anra_paths import inject_all_paths, get_v2_checkpoint
inject_all_paths()
from training.v2_runtime import sync_to_drive, _read_step

# Fixed artifacts only. These calls overwrite the same canonical filenames in
# Drive/AnRa and Drive/AnRa/v2/checkpoints; no step-suffixed files are created.
sync_to_drive("brain")
sync_to_drive("tokenizer")
sync_to_drive("identity")
sync_to_drive("ouroboros")
sync_to_drive("eval_summary")

# Sync scorecards/reports as logs, not model checkpoints.
os.makedirs(DRIVE_REPORTS, exist_ok=True)
report_sources = [repo / "output" / "v2" / "reports", repo / "output" / "v2" / "eval"]
for folder in report_sources:
    if folder.is_dir():
        for fpath in folder.glob("*.json"):
            dest = Path(DRIVE_REPORTS) / fpath.name
            shutil.copy2(fpath, dest)
            print("report", fpath.name)

scorecard_local = repo / "output" / "v2" / "scorecards"
drive_scorecards = Path(DRIVE_V2_DIR) / "scorecards"
if scorecard_local.is_dir():
    drive_scorecards.mkdir(parents=True, exist_ok=True)
    for sc_file in scorecard_local.glob("*.json"):
        dest = drive_scorecards / sc_file.name
        shutil.copy2(sc_file, dest)
        print(f"Scorecard synced: {sc_file.name}")
else:
    print("No local scorecards to sync")

# Sync training data back if it grew (e.g. from self-play or new conversations)
local_data = repo / "training_data" / "anra_training.txt"
if local_data.exists():
    current_size = local_data.stat().st_size
    drive_data = Path(DRIVE_ANRA_DIR) / "anra_training.txt"
    drive_size = drive_data.stat().st_size if drive_data.exists() else 0
    if current_size > drive_size + 1024:
        shutil.copy2(local_data, drive_data)
        print(f"Training data updated: {drive_size//1024}KB -> {current_size//1024}KB")
    else:
        print("Training data unchanged")

# Sync HAL state
hal_local = repo / "state" / "hal_state.json"
if hal_local.exists():
    hal_drive = Path(DRIVE_ANRA_DIR) / "logs" / "hal_state.json"
    hal_drive.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(hal_local, hal_drive)
    print("HAL state synced to Drive")

brain_path = get_v2_checkpoint("brain")
if brain_path.exists():
    print(f"Brain checkpoint step: {_read_step(brain_path)}")

print("Drive reports:", DRIVE_REPORTS)
print("Drive checkpoints:", DRIVE_V2_CHECKPOINTS)
print("Safe to close Colab now.")


In [ ]:
# CELL 10 - API OR UI LAUNCH
# Optional. Run after repo setup. The CLI report/training cells above are the primary operator path.
import os
import sys
import subprocess

REPO_PATH = open("/tmp/anra_repo_path.txt", encoding="utf-8").read().strip()
print("Repo:", REPO_PATH)
print("Available launch options:")
print("1. CLI status:   !python anra.py --status")
print("2. CLI report:   !python anra.py --report")
print("3. FastAPI app:   python app.py or uvicorn app:app --host 0.0.0.0 --port 8000")
print("4. Legacy UI:     run ui/anra_ui_launcher.py if present")

launcher = f"{REPO_PATH}/ui/anra_ui_launcher.py"
if os.path.exists(launcher):
    run_ui = input("Launch legacy UI now? [y/N]: ").strip().lower() == "y"
    if run_ui:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flask", "flask-cors"], capture_output=True)
        exec(open(launcher, encoding="utf-8").read())
else:
    print("Legacy UI launcher not found; use anra.py or app.py surfaces.")
